# Upload local DistilBERT (or any HF model) files to S3

This notebook recursively uploads a local Hugging Face model directory to S3 so it can be used in **offline mode** by the SageMaker DistilBERT training notebook.

**What you need locally**
- A folder with Hugging Face model files (e.g., `config.json`, `pytorch_model.bin`, `tokenizer.json` or `vocab.txt`/`merges.txt`, `tokenizer_config.json`, etc.).

**How to use**
1. Set the `CONFIG` values below (`LOCAL_MODEL_DIR`, `S3_BUCKET`, `MODEL_S3_PREFIX`).
2. Run all cells. The model files will be uploaded to `s3://<bucket>/<MODEL_S3_PREFIX>`.

**Tip**: You can use this for any HF model, not just DistilBERT.

In [ ]:
#!pip install -q boto3
import os
import boto3
from datetime import datetime


In [ ]:
# =====================
# CONFIG — EDIT ME
# =====================
CONFIG = {
    'AWS_REGION': os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'),
    'LOCAL_MODEL_DIR': '/path/to/distilbert-base-uncased',  # <-- change to your local folder
    'S3_BUCKET': 'your-bucket-name',                        # <-- change
    'MODEL_S3_PREFIX': 'models/distilbert-base-uncased/',   # S3 prefix to upload INTO
}
CONFIG

In [ ]:
s3 = boto3.client('s3', region_name=CONFIG['AWS_REGION'])

def s3_uri(bucket, key):
    return f's3://{bucket}/{key}'

def upload_dir(local_dir, bucket, prefix):
    local_dir = os.path.abspath(local_dir)
    uploaded = []
    for root, dirs, files in os.walk(local_dir):
        for f in files:
            lp = os.path.join(root, f)
            rel = os.path.relpath(lp, local_dir).replace('\\','/')
            key = prefix.rstrip('/') + '/' + rel
            s3.upload_file(lp, bucket, key)
            uploaded.append(s3_uri(bucket, key))
    return uploaded


In [ ]:
# Basic checks
assert os.path.isdir(CONFIG['LOCAL_MODEL_DIR']), f"LOCAL_MODEL_DIR not found: {CONFIG['LOCAL_MODEL_DIR']}"
print('Local model files:')
for root, dirs, files in os.walk(CONFIG['LOCAL_MODEL_DIR']):
    for f in files[:5]:
        print(' ', os.path.join(root, f))
    break

In [ ]:
# Upload now
uploaded_uris = upload_dir(CONFIG['LOCAL_MODEL_DIR'], CONFIG['S3_BUCKET'], CONFIG['MODEL_S3_PREFIX'])
print(f'Uploaded {len(uploaded_uris)} files. Example:')
print('\n'.join(uploaded_uris[:10]))